rain/validation/test split

In [1]:
import random 
import math 


def train_val_test_split(X,y,train_ratio=0.6,val_ratio=0.2,seed=42):
    random.seed(seed)
    n=len(X)
    indices=list(range(n))
    random.shuffle(indices)
    
    train_end=int(n*train_ratio)
    val_end=int(n*(train_ratio+val_ratio))
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]

    X_train = [X[i] for i in train_idx]
    y_train = [y[i] for i in train_idx]
    X_val = [X[i] for i in val_idx]
    y_val = [y[i] for i in val_idx]
    X_test = [X[i] for i in test_idx]
    y_test = [y[i] for i in test_idx]

    return X_train, y_train, X_val, y_val, X_test, y_test

In [3]:
X = list(range(10))
y = [x * x for x in X]

X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(X, y)
print(X,y)
print("Train X:", X_train)
print("Train y:", y_train)
print("Val   X:", X_val)
print("Val   y:", y_val)
print("Test  X:", X_test)
print("Test  y:", y_test)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9] [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
Train X: [7, 3, 2, 8, 5, 6]
Train y: [49, 9, 4, 64, 25, 36]
Val   X: [9, 4]
Val   y: [81, 16]
Test  X: [0, 1]
Test  y: [0, 1]


K-fold and stratified K-fold cross-validation

In [5]:
def kfold_split(n, k=5, seed=42):
    random.seed(seed)
    indices = list(range(n))
    random.shuffle(indices)

    fold_size = n // k
    folds = []

    for i in range(k):
        start = i * fold_size
        end = start + fold_size if i < k - 1 else n
        val_idx = indices[start:end]
        train_idx = indices[:start] + indices[end:]
        folds.append((train_idx, val_idx))

    return folds


def stratified_kfold_split(y, k=5, seed=42):
    random.seed(seed)

    class_indices = {}
    for i, label in enumerate(y):
        class_indices.setdefault(label, []).append(i)

    for label in class_indices:
        random.shuffle(class_indices[label])

    folds = [{"train": [], "val": []} for _ in range(k)]

    for label, indices in class_indices.items():
        fold_size = len(indices) // k
        for i in range(k):
            start = i * fold_size
            end = start + fold_size if i < k - 1 else len(indices)
            val_part = indices[start:end]
            train_part = indices[:start] + indices[end:]
            folds[i]["val"].extend(val_part)
            folds[i]["train"].extend(train_part)

    return [(f["train"], f["val"]) for f in folds]


def cross_validate(X, y, model_fn, k=5, metric_fn=None, stratified=False):
    n = len(X)

    if stratified:
        folds = stratified_kfold_split(y, k)
    else:
        folds = kfold_split(n, k)

    scores = []
    for train_idx, val_idx in folds:
        X_train = [X[i] for i in train_idx]
        y_train = [y[i] for i in train_idx]
        X_val = [X[i] for i in val_idx]
        y_val = [y[i] for i in val_idx]

        model = model_fn()
        model.fit(X_train, y_train)
        predictions = [model.predict(x) for x in X_val]

        if metric_fn:
            score = metric_fn(y_val, predictions)
        else:
            score = sum(1 for yt, yp in zip(y_val, predictions) if yt == yp) / len(y_val)
        scores.append(score)

    return scores

In [6]:
folds = kfold_split(n=10, k=5, seed=42)
assert len(folds) == 5
for train_idx, val_idx in folds:
    assert len(val_idx) == 2
    assert len(train_idx) == 8
    assert set(train_idx).isdisjoint(val_idx)
all_val = [i for _, val in folds for i in val]
assert sorted(all_val) == list(range(10))
assert kfold_split(10, 5, seed=42) == folds

In [7]:
y = [0] * 6 + [1] * 4
folds = stratified_kfold_split(y, k=2, seed=42)
assert len(folds) == 2
for train_idx, val_idx in folds:
    assert set(train_idx).isdisjoint(val_idx)
    labels = [y[i] for i in val_idx]
    assert 0 in labels and 1 in labels
all_val = [i for _, val in folds for i in val]
assert sorted(all_val) == list(range(len(y)))

In [8]:
X = [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9]]
y = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]

class ThresholdModel:
    def fit(self, X, y):
        self.threshold = sum(x[0] for x in X) / len(X)
    def predict(self, x):
        return 1 if x[0] > self.threshold else 0

def accuracy(a, b):
    return sum(x == y for x, y in zip(a, b)) / len(a)

scores = cross_validate(X, y, lambda: ThresholdModel(), k=5, metric_fn=accuracy)
assert len(scores) == 5
assert all(0.0 <= s <= 1.0 for s in scores)
assert scores == cross_validate(X, y, lambda: ThresholdModel(), k=5, metric_fn=accuracy)